In [9]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import pandas as pd
import datetime
from datetime import date
import random
import subprocess
import time as t
import csv
import re
import os

link = "https://www.southernrailway.com"

leaving_from = "Clapham Junction"
going_to = "Brighton"

In [ ]:



driver = webdriver.Edge()
driver.get(link)
t.sleep(5)

button = driver.find_element(By.ID, "CybotCookiebotDialogBodyLevelButtonLevelOptinAllowAll")
button.click()
t.sleep(3)

shadow_host = driver.find_element(By.ID, "otrl-custom-hero")
shadow_root = shadow_host.shadow_root
leaving =   shadow_root.find_element(By.NAME, "stationFrom")
leaving.send_keys(leaving_from)
WebDriverWait(driver, 2).until(
    lambda d: len(shadow_root.find_elements(
        By.CSS_SELECTOR, ".otrl-jp__station-autosuggest__item"
    )) > 0
)
first_suggestion = shadow_root.find_element(
    By.CSS_SELECTOR, ".otrl-jp__station-autosuggest__item"
)
first_suggestion.click()

t.sleep(3)
going = shadow_root.find_element(By.NAME, "stationTo")
going.send_keys(going_to)
WebDriverWait(driver, 2).until(
    lambda d: len(shadow_root.find_elements(
        By.CSS_SELECTOR, ".otrl-jp__station-autosuggest__item"
    )) > 0
)
first_suggestion = shadow_root.find_element(
    By.CSS_SELECTOR, ".otrl-jp__station-autosuggest__item"
)
first_suggestion.click()


t.sleep(3)

search = shadow_root.find_element(By.CLASS_NAME, "otrl-jp__tickets__submit")
search.click()
t.sleep(5)

listview = driver.find_element(By.CSS_SELECTOR, 'a[aria-label="List view"]')
listview.click()
t.sleep(5)

wait = WebDriverWait(driver, 2)
seen_departures = set()
records = []

try:
    date_text = driver.find_element(By.CSS_SELECTOR, ".service-list__heading2").text.strip()  # e.g. "Wed 04 Mar 2026"
    current_date = datetime.datetime.strptime(date_text, "%a %d %b %Y").date()
except Exception as e:
    print(f"  Could not parse date: {e}")

current_date_tracker = current_date


def parse_current_page(current_date_tracker):
    """Parse all visible trains and fares from list view, skip already-seen departures."""

    cards = driver.find_elements(By.CSS_SELECTOR, ".service-list__card.service-fare")

    for i, card in enumerate(cards):
        try:
            # Journey info from sr-text h4
            sr_text = card.find_element(By.CSS_SELECTOR, "h4.sr-text").text

            dep_match = re.search(r'^(\d{2}:\d{2})', sr_text)
            arr_match = re.search(r'arriving at .+? at (\d{2}:\d{2})', sr_text)
            dur_match = re.search(r'takes (.+?),', sr_text)
            chg_match = re.search(r'has (\d+) change', sr_text)

            departure = dep_match.group(1) if dep_match else None
            if not departure:
                continue

            arrival  = arr_match.group(1) if arr_match else None
            dep_time = datetime.datetime.strptime(departure, "%H:%M").time()
            current_date = current_date_tracker

            if records:
                last_dep_dt = records[-1]["departure_dt"]
                candidate_dt = datetime.datetime.combine(current_date, dep_time)
                if (last_dep_dt - candidate_dt).total_seconds() > 6 * 3600:
                    current_date = current_date + datetime.timedelta(days=1)
                    current_date_tracker = current_date
                    print(f"  Date rolled over to {current_date}")

            dedup_key = (current_date, departure)
            if dedup_key in seen_departures:
                continue

            duration = dur_match.group(1).strip() if dur_match else None
            changes  = int(chg_match.group(1)) if chg_match else 0

            # Price from the continue button — contained within the same card
            price = None
            try:
    # Target the sr-text span inside the button for a clean price string
                price_element = card.find_element(By.CSS_SELECTOR, ".btn-continue .sr-text")
                price_text = price_element.text  
    
    # Clean the string (remove £) and convert to float
                price = float(price_text.replace('£', '').strip())
            except Exception:
                pass  # No fare available for this service

            departure_dt = datetime.datetime.combine(current_date, dep_time)
            arrival_dt   = None
            if arrival:
                arr_time = datetime.datetime.strptime(arrival, "%H:%M").time()
                arrival_dt = datetime.datetime.combine(current_date, arr_time)
                if arrival_dt < departure_dt:
                    arrival_dt += datetime.timedelta(days=1)

            records.append({
                "departure_dt": departure_dt,
                "arrival_dt":   arrival_dt,
                "departure":    departure,
                "arrival":      arrival,
                "duration":     duration,
                "changes":      changes,
                "price_gbp":    price,
            })
            seen_departures.add(dedup_key)

        except Exception as e:
            print(f"  Error parsing traincard {i}: {e}")


parse_current_page(current_date_tracker)


for click_num in range(15):
    try:
        t.sleep(random.uniform(0.3, 1.4))

        time_elements = driver.find_elements(
            By.CSS_SELECTOR, ".service-list-v2__services li .service-summary__station time"
        )
        if not time_elements:
            print(f"Click {click_num + 1}: no time elements found before click, skipping.")
            break
        last_departure_before = time_elements[-1].get_attribute("datetime")

        later_btn = wait.until(EC.element_to_be_clickable(
            (By.CSS_SELECTOR, "a.service-pager[aria-label='Show later trains']")
        ))
        driver.execute_script("arguments[0].click();", later_btn)

        def page_has_updated(d):
            try:
                elements = d.find_elements(
                    By.CSS_SELECTOR, ".service-list-v2__services li .service-summary__station time"
                )
                return (
                    len(elements) > 0
                    and elements[-1].get_attribute("datetime") != last_departure_before
                )
            except Exception:
                return False

        WebDriverWait(driver, 10).until(page_has_updated)

        # Wait for fare buttons to be rendered before scraping prices
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, ".service-list-v2__services li .btn-continue"))
        )

    except TimeoutException:
        print(f"Click {click_num + 1}: page didn't update or no 'Later' button.")
        break

    parse_current_page(current_date_tracker)
    print(f"After click {click_num + 1}: {len(records)} trains total")


driver.quit()
df = pd.DataFrame(records)
print(df.head())

          departure_dt          arrival_dt departure arrival  \
0  2026-03-04 00:09:00 2026-03-04 01:16:00     00:09   01:16   
1  2026-03-04 00:30:00 2026-03-04 02:28:00     00:30   02:28   
2  2026-03-04 04:42:00 2026-03-04 06:10:00     04:42   06:10   
3  2026-03-04 05:27:00 2026-03-04 06:43:00     05:27   06:43   
4  2026-03-04 05:41:00 2026-03-04 06:59:00     05:41   06:59   
..                 ...                 ...       ...     ...   
58 2026-03-04 19:12:00 2026-03-04 20:18:00     19:12   20:18   
59 2026-03-04 19:31:00 2026-03-04 20:27:00     19:31   20:27   
60 2026-03-04 19:42:00 2026-03-04 20:48:00     19:42   20:48   
61 2026-03-04 20:01:00 2026-03-04 20:57:00     20:01   20:57   
62 2026-03-04 20:12:00 2026-03-04 21:18:00     20:12   21:18   

             duration  changes  price_gbp  
0    1 hour 7 minutes        0       33.2  
1   1 hour 58 minutes        1       33.2  
2   1 hour 28 minutes        1       33.2  
3   1 hour 16 minutes        1       33.2  
4   1 hour 